# What does your voice look like?

In [1]:
import sounddevice as sd
import numpy as np
import librosa
import librosa.display
import matplotlib.pyplot as plt
from IPython.display import Audio, display

duration = 5  # seconds
sample_rate = 16000

print(f"🎤 Recording for {duration} seconds... Speak now!")
audio = sd.rec(int(duration * sample_rate), 
               samplerate=sample_rate, 
               channels=1, 
               dtype='float32')
sd.wait()
audio = audio.flatten()
print("✅ Recording complete!")


print("This is your voice")
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

time = np.linspace(0, len(audio) / sample_rate, len(audio))
axes[0].plot(time, audio, color='blue', linewidth=0.5)
axes[0].set_title('Your Voice Waveform', fontsize=14)
axes[0].set_xlabel('Time (seconds)')
axes[0].set_ylabel('Amplitude')
axes[0].grid(True, alpha=0.3)

print("Mel Spectogram") 
mel_spec = librosa.feature.melspectrogram(y=audio, sr=sample_rate, n_mels=80)
mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
img = librosa.display.specshow(mel_spec_db, sr=sample_rate, 
                                x_axis='time', y_axis='mel', 
                                ax=axes[1], cmap='viridis')
axes[1].set_title('Mel Spectrogram - Your Voice as a Picture!', fontsize=14)
plt.colorbar(img, ax=axes[1], format='%+2.0f dB')

print("Spectogram") 
D = librosa.stft(audio)
D_db = librosa.amplitude_to_db(np.abs(D), ref=np.max)
img2 = librosa.display.specshow(D_db, sr=sample_rate, 
                                 x_axis='time', y_axis='hz', 
                                 ax=axes[2], cmap='magma')
axes[2].set_title('Frequency Spectrogram', fontsize=14)
axes[2].set_ylim(0, 4000)  # Focus on speech frequencies
plt.colorbar(img2, ax=axes[2], format='%+2.0f dB')

plt.tight_layout()
plt.show()

ModuleNotFoundError: No module named 'sounddevice'

# Whisper API

In [ ]:
import scipy.io.wavfile as wavfile
from openai import OpenAI
import io
import os
from IPython.display import Audio, display
from dotenv import load_dotenv

load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

duration = 5  # seconds
sample_rate = 16000

print(f"🎤 Recording for {duration} seconds... Speak now!")
audio = sd.rec(int(duration * sample_rate), 
               samplerate=sample_rate, 
               channels=1, 
               dtype='float32')
sd.wait()
audio = audio.flatten()
print("✅ Recording complete!")


audio_int16 = np.int16(audio * 32767)

# Create in-memory WAV file
buffer = io.BytesIO()
wavfile.write(buffer, sample_rate, audio_int16)
buffer.seek(0)
buffer.name = "recording.wav"  # Whisper needs a filename

# Transcribe with Whisper
print("🤖 Transcribing with Whisper...")
transcript = client.audio.transcriptions.create(
    model="whisper-1",
    file=buffer
)

print("\n📝 Transcription:")
print("-" * 40)
print(transcript.text)

# Super Power: Context

In [ ]:
import sounddevice as sd
import numpy as np
import scipy.io.wavfile as wavfile
from openai import OpenAI
import io
from IPython.display import Audio, display, Markdown

#client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# Display a 30-second reading text
reading_text = """
**Read this text aloud (about 30 seconds):**

The researchers' analyses of the wind farm data showed variable wind patterns. The project 
lead will lead the team through complex permit processes. They must present the present 
findings to investors. The 3 million contract includes clauses for force majeure 
events. Dr. García-Smith's team discovered that the enzyme's pH of 7.2 was optimal. The 
AI learned to differentiate between "read" (present) and "read" (past), processing live 
feeds versus archived content. The baroque bass player stood close to the entrance, too 
close to record properly.
"""

display(Markdown(reading_text))

# Record for 30 seconds
duration = 45
sample_rate = 16000

print(f"\n🎤 Recording for {duration} seconds... Start reading now!")
audio = sd.rec(int(duration * sample_rate), samplerate=sample_rate, channels=1, dtype='float32')
sd.wait()
audio = audio.flatten()
print("✅ Recording complete!")

display(Audio(audio, rate=sample_rate))


# Prepare audio
audio_int16 = np.int16(audio * 32767)
buffer = io.BytesIO()
wavfile.write(buffer, sample_rate, audio_int16)
buffer.seek(0)
buffer.name = "recording.wav"

# Transcribe without context prompt
print("🤖 Transcribing without context...")
transcript = client.audio.transcriptions.create(
    model="whisper-1",
    file=buffer,
)

print("\n📝 Transcription:")
print("-" * 40)
print(transcript.text)

In [ ]:

# Transcribe with context prompt
print("🤖 Transcribing with context...")
transcript = client.audio.transcriptions.create(
    model="whisper-1",
    file=buffer,
    prompt = "heteronyms, homographs, lead metal, wind turbine, present tense, past tense read, pH levels, enzyme, García-Smith, force majeure, baroque, bass instrument, differentiate")

print("\n📝 Transcription:")
print("-" * 40)
print(transcript.text)


# Chunking

In [ ]:
import sounddevice as sd
import numpy as np
import scipy.io.wavfile as wavfile
from openai import OpenAI
import io

#client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# Record longer audio
duration = 15  # seconds total
sample_rate = 16000
chunk_duration = 5  # seconds per chunk

print(f"🎤 Recording for {duration} seconds... Speak continuously!")
audio = sd.rec(int(duration * sample_rate), samplerate=sample_rate, channels=1, dtype='float32')
sd.wait()
audio = audio.flatten()
print("✅ Recording complete!")

display(Audio(audio, rate=sample_rate))


# Split audio into chunks
chunk_size = chunk_duration * sample_rate
chunks = [audio[i:i+chunk_size] for i in range(0, len(audio), chunk_size)]

print(f"\n🔪 Split into {len(chunks)} chunks")

# Transcribe each chunk
all_transcripts = []

for i, chunk in enumerate(chunks):
    print(f"\n🤖 Transcribing chunk {i+1}/{len(chunks)}...")
    
    # Prepare chunk buffer
    chunk_int16 = np.int16(chunk * 32767)
    buffer = io.BytesIO()
    wavfile.write(buffer, sample_rate, chunk_int16)
    buffer.seek(0)
    buffer.name = f"chunk_{i}.wav"
    
    # Transcribe
    transcript = client.audio.transcriptions.create(
        model="whisper-1",
        file=buffer
    )
    
    all_transcripts.append(transcript.text)
    print(f"Chunk {i+1}: {transcript.text}")

# Combine all transcripts
print("\n📝 Complete Transcription:")
print("-" * 40)
full_text = " ".join(all_transcripts)
print(full_text)

# Time Stamps

In [ ]:
import sounddevice as sd
import numpy as np
import scipy.io.wavfile as wavfile
from openai import OpenAI
import io
from IPython.display import Audio, display

#client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

duration = 10  # seconds
sample_rate = 16000

print(f"🎤 Recording for {duration} seconds... Speak with pauses!")
audio = sd.rec(int(duration * sample_rate), samplerate=sample_rate, channels=1, dtype='float32')
sd.wait()
audio = audio.flatten()
print("✅ Recording complete!")

# Play back
display(Audio(audio, rate=sample_rate))

# Prepare audio
audio_int16 = np.int16(audio * 32767)
buffer = io.BytesIO()
wavfile.write(buffer, sample_rate, audio_int16)
buffer.seek(0)
buffer.name = "recording.wav"

# Get detailed transcription with timestamps
print("\n🤖 Transcribing with timestamps...")
transcript = client.audio.transcriptions.create(
    model="whisper-1",
    file=buffer,
    response_format="verbose_json",
    timestamp_granularities=["segment", "word"]  # Get both segment and word timestamps
)

# Display formatted timestamps
print("\n📝 Transcription with timestamps:")
print("-" * 40)

# Show segments with timestamps
if hasattr(transcript, 'segments'):
    for segment in transcript.segments:
        start = segment.start
        end = segment.end
        text = segment.text
        print(f"\n[{start:.2f}s - {end:.2f}s]")
        print(f"  {text}")

# Show word-level timestamps if available
if hasattr(transcript, 'words'):
    print("\n🔤 Word-level timing:")
    print("-" * 40)
    for word in transcript.words:
        word_text = word.word
        start = word.start
        end = word.end
        print(f"{word_text:15} [{start:.2f}s - {end:.2f}s]")

# Full text and metadata
print("\n📄 Full text:")
print(transcript.text)

print(f"\n🌍 Language detected: {transcript.language}")
print(f"⏱️ Total duration: {transcript.duration:.2f}s")

Step 3: Basic Transcription (Without Chunking)


In [2]:
from pathlib import Path
from openai import OpenAI

client = OpenAI()

audio_file = next(Path("audio").glob("*.mp3"))

with audio_file.open("rb") as file:
    transcription = client.audio.transcriptions.create(
        model="whisper-1",
        file=file
    )

print("Response type:", type(transcription))
print("Full response:", transcription)
print("\nExtracted text:\n")
print(transcription.text)

Response type: <class 'openai.types.audio.transcription.Transcription'>
Full response: Transcription(text="So, we decided we had to put in a new well, and somebody asked me why I didn't get it witched. I didn't know what he was talking about, so they said, there's an old fella out here that witches wells, and he'll get you water every time. So I said, okay, how much does he want? He wants ten bucks. I said, bring him on. So we brought this fella out, and he went through his maneuvers with his peach stick, or I don't remember now if it was a peach stick or a well. You used both though. Yeah. One works as good as the other, and he strikes a point where he hits water, and he puts a stone down. Then he goes down in that same line, and where he hits water again, he puts another stone down. Then he goes at right angles with this, and he gets a stone over here and a stone over here, and where these two lines cross, that's the point. The veins, he said these water veins always run on an angle.

In [3]:
from pathlib import Path

output_path = Path("transcripts/basic_transcription.txt")

output_path.write_text(
    transcription.text,
    encoding="utf-8"
)

print(f"Saved to: {output_path}")

Saved to: transcripts\basic_transcription.txt


In [4]:
print(output_path.exists())
print(output_path.read_text(encoding="utf-8"))

True
So, we decided we had to put in a new well, and somebody asked me why I didn't get it witched. I didn't know what he was talking about, so they said, there's an old fella out here that witches wells, and he'll get you water every time. So I said, okay, how much does he want? He wants ten bucks. I said, bring him on. So we brought this fella out, and he went through his maneuvers with his peach stick, or I don't remember now if it was a peach stick or a well. You used both though. Yeah. One works as good as the other, and he strikes a point where he hits water, and he puts a stone down. Then he goes down in that same line, and where he hits water again, he puts another stone down. Then he goes at right angles with this, and he gets a stone over here and a stone over here, and where these two lines cross, that's the point. The veins, he said these water veins always run on an angle. Never do north and south or east and west, or north, west, south, east.


Step 4: Transcription with Prompts (Guided Approach)

In [5]:
from pathlib import Path

audio_file = next(Path("audio").glob("*.mp3"))

context_prompt = """
This is a conversational oral-history recording about well witching and finding groundwater.
Relevant terms may include water veins, peach stick, wells, right angles,
north, south, east, and west. Preserve the natural conversational wording.
"""

with audio_file.open("rb") as file:
    prompted_transcription = client.audio.transcriptions.create(
        model="whisper-1",
        file=file,
        prompt=context_prompt
    )

print(prompted_transcription.text)

So we decided we had to put in a new well, and somebody asked me why I didn't get it witched, and I didn't know what he was talking about, so they said there's an old fella out here that witches wells, and he'll get you water every time. So I said, okay, how much does he want? He wants ten bucks. I said, bring him on. So we brought this fella out, and he went through his maneuvers with his peach stick, or I don't remember now if it was a peach stick or a well. You used both, though. Yeah, either one. One works as good as the other. And he strikes a point where he hits water, and he puts a stone down. Then he goes down in that same line, and where he hits water again, he puts another stone down. Then he goes at right angles with this, and he gets a stone over here and a stone over here, and where these two lines cross, that's the point. The veins, he said these water veins always run on an angle. Never do north and south or east and west. They're north, west, south, east.


In [6]:
prompted_output_path = Path("transcripts/prompted_transcription.txt")

prompted_output_path.write_text(
    prompted_transcription.text,
    encoding="utf-8"
)

print(f"Saved to: {prompted_output_path}")

Saved to: transcripts\prompted_transcription.txt


In [7]:
print(prompted_output_path.exists())

True


step 5 Transcription without prompt (unguided approach)

In [8]:
from pathlib import Path

unprompted_output_path = Path("transcripts/unprompted_transcription.txt")

unprompted_output_path.write_text(
    transcription.text,
    encoding="utf-8"
)

print(f"Saved to: {unprompted_output_path}")

Saved to: transcripts\unprompted_transcription.txt


In [9]:
from difflib import SequenceMatcher

prompted_text = prompted_output_path.read_text(encoding="utf-8")
unprompted_text = unprompted_output_path.read_text(encoding="utf-8")

similarity = SequenceMatcher(
    None,
    unprompted_text,
    prompted_text
).ratio()

print(f"Similarity: {similarity:.1%}")
print("\n--- UNPROMPTED ---\n")
print(unprompted_text)
print("\n--- PROMPTED ---\n")
print(prompted_text)

Similarity: 89.2%

--- UNPROMPTED ---

So, we decided we had to put in a new well, and somebody asked me why I didn't get it witched. I didn't know what he was talking about, so they said, there's an old fella out here that witches wells, and he'll get you water every time. So I said, okay, how much does he want? He wants ten bucks. I said, bring him on. So we brought this fella out, and he went through his maneuvers with his peach stick, or I don't remember now if it was a peach stick or a well. You used both though. Yeah. One works as good as the other, and he strikes a point where he hits water, and he puts a stone down. Then he goes down in that same line, and where he hits water again, he puts another stone down. Then he goes at right angles with this, and he gets a stone over here and a stone over here, and where these two lines cross, that's the point. The veins, he said these water veins always run on an angle. Never do north and south or east and west, or north, west, south, e

Step 6: Implementing Audio Chunking

In [10]:
from pathlib import Path
from pydub import AudioSegment
from pydub.utils import make_chunks

audio_file = next(Path("audio").glob("*.mp3"))
audio = AudioSegment.from_file(audio_file)

chunk_length_ms = 30 * 1000
audio_chunks = make_chunks(audio, chunk_length_ms)

chunk_paths = []

for index, chunk in enumerate(audio_chunks):
    chunk_path = Path("chunks") / f"chunk_{index:03d}.mp3"
    chunk.export(chunk_path, format="mp3")
    chunk_paths.append(chunk_path)

    print(
        f"{chunk_path.name}: "
        f"{len(chunk) / 1000:.1f} seconds"
    )

print(f"Created {len(chunk_paths)} chunk(s).")

chunk_000.mp3: 30.0 seconds
chunk_001.mp3: 30.0 seconds
chunk_002.mp3: 6.7 seconds
Created 3 chunk(s).


Verify the files:

In [11]:
list(Path("chunks").glob("*.mp3"))

[WindowsPath('chunks/chunk_000.mp3'),
 WindowsPath('chunks/chunk_001.mp3'),
 WindowsPath('chunks/chunk_002.mp3')]

Step 7: Transcribing Chunks with Timestamps


Transcribe each chunk with segment timestamps

In [12]:
from pathlib import Path
import json

chunk_paths = sorted(Path("chunks").glob("*.mp3"))
chunk_length_seconds = 30

all_segments = []

for index, chunk_path in enumerate(chunk_paths):
    chunk_offset = index * chunk_length_seconds

    print(f"Transcribing {chunk_path.name}...")

    with chunk_path.open("rb") as file:
        result = client.audio.transcriptions.create(
            model="whisper-1",
            file=file,
            response_format="verbose_json",
            timestamp_granularities=["segment"]
        )

    for segment in result.segments:
        local_start = float(segment.start)
        local_end = float(segment.end)

        all_segments.append({
            "chunk": chunk_path.name,
            "start": chunk_offset + local_start,
            "end": chunk_offset + local_end,
            "text": segment.text.strip()
        })

print(f"Collected {len(all_segments)} timestamped segments.")

Transcribing chunk_000.mp3...
Transcribing chunk_001.mp3...
Transcribing chunk_002.mp3...
Collected 20 timestamped segments.


The important part is:

In [13]:
chunk_offset + local_start

60.0

 Save the combined timestamped result

In [14]:
json_path = Path("transcripts/chunked_segments.json")

json_path.write_text(
    json.dumps(all_segments, indent=2),
    encoding="utf-8"
)

print(f"Saved to {json_path}")

Saved to transcripts\chunked_segments.json


Inspect the first few segments:

In [15]:
for segment in all_segments[:5]:
    print(
        f"{segment['start']:.2f}s - "
        f"{segment['end']:.2f}s: "
        f"{segment['text']}"
    )

0.00s - 6.16s: So, we decided we had to put in a new well, and somebody asked me why I didn't get it
6.16s - 7.16s: which.
7.16s - 9.64s: And I didn't know what he was talking about, so they said, there's an old fella out here
9.64s - 13.76s: that which his wells, and he'll get you water every time.
13.76s - 17.04s: So I said, okay, how much does he want?


Step 8: Exporting with Timestamps
Export transcriptions in readable formats with timestamps.
We’ll now create three readable exports from all_segments:
- Human-readable TXT
- SRT subtitle file
- JSON with metadata and segments

In [16]:
from pathlib import Path
import json

export_dir = Path("exports")
export_dir.mkdir(exist_ok=True)

def format_timestamp(seconds, separator="."):
    milliseconds = round(float(seconds) * 1000)

    hours, remainder = divmod(milliseconds, 3_600_000)
    minutes, remainder = divmod(remainder, 60_000)
    seconds_value, milliseconds = divmod(remainder, 1_000)

    return (
        f"{hours:02d}:{minutes:02d}:{seconds_value:02d}"
        f"{separator}{milliseconds:03d}"
    )

source_audio = next(Path("audio").glob("*.mp3")).name

Export human-readable TXT

In [17]:
txt_lines = []

for index, segment in enumerate(all_segments, start=1):
    start = format_timestamp(segment["start"])
    end = format_timestamp(segment["end"])

    txt_lines.append(
        f"[{start} --> {end}] {segment['text']}"
    )

txt_path = export_dir / "chunked_transcription.txt"
txt_path.write_text(
    "\n".join(txt_lines),
    encoding="utf-8"
)

print(f"Saved: {txt_path}")

Saved: exports\chunked_transcription.txt


Export SRT

In [18]:
srt_blocks = []

for index, segment in enumerate(all_segments, start=1):
    start = format_timestamp(segment["start"], separator=",")
    end = format_timestamp(segment["end"], separator=",")

    srt_blocks.append(
        f"{index}\n"
        f"{start} --> {end}\n"
        f"{segment['text']}\n"
    )

srt_path = export_dir / "chunked_transcription.srt"
srt_path.write_text(
    "\n".join(srt_blocks),
    encoding="utf-8"
)

print(f"Saved: {srt_path}")

Saved: exports\chunked_transcription.srt


Export JSON

In [19]:
json_export = {
    "source_audio": source_audio,
    "segment_count": len(all_segments),
    "segments": all_segments
}

json_path = export_dir / "chunked_transcription.json"
json_path.write_text(
    json.dumps(json_export, indent=2),
    encoding="utf-8"
)

print(f"Saved: {json_path}")

Saved: exports\chunked_transcription.json


Verify the exports

In [20]:
for path in sorted(export_dir.iterdir()):
    print(path.name, "-", path.stat().st_size, "bytes")

chunked_transcription.json - 3677 bytes
chunked_transcription.srt - 1770 bytes
chunked_transcription.txt - 1679 bytes


Read the beginning of the text and SRT files:

In [21]:
print(txt_path.read_text(encoding="utf-8")[:1000])
print(srt_path.read_text(encoding="utf-8")[:1000])

[00:00:00.000 --> 00:00:06.160] So, we decided we had to put in a new well, and somebody asked me why I didn't get it
[00:00:06.160 --> 00:00:07.160] which.
[00:00:07.160 --> 00:00:09.640] And I didn't know what he was talking about, so they said, there's an old fella out here
[00:00:09.640 --> 00:00:13.760] that which his wells, and he'll get you water every time.
[00:00:13.760 --> 00:00:17.040] So I said, okay, how much does he want?
[00:00:17.040 --> 00:00:18.040] He wants ten bucks.
[00:00:18.040 --> 00:00:19.320] I said, bring him on.
[00:00:19.320 --> 00:00:24.680] So we brought this fella out, and he went through his maneuvers with his peach stick,
[00:00:24.680 --> 00:00:27.960] or I don't remember now if it was a peach stick or a well.
[00:00:27.960 --> 00:00:28.960] You used both though.
[00:00:28.960 --> 00:00:29.960] Yeah.
[00:00:29.960 --> 00:00:30.960] Yeah.
[00:00:30.960 --> 00:00:31.960] Yeah.
[00:00:31.960 --> 00:00:32.960] Yeah.
[00:00:32.960 --> 00:00:32.960] 
[00:00